In [38]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn.model_selection import train_test_split
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sksurv.metrics import integrated_brier_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

In [39]:
dataset = pd.read_csv("data_treated.csv")

treatment_variables = []
with open("diagnosis.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

diagnosis_variables = []
with open("treatment.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [40]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [41]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [42]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

sgkf = StratifiedKFold(n_splits=3, shuffle=True, random_state=12)

In [23]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [44]:
param_grid = {
    "learning_rate": [0.01, 0.1, 0.2],
    "n_estimators": [100, 200],
    "max_depth": [1, 3, 5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 5]
}

cv = sgkf.split(X=x_train, y=y_train['event'])
gbm = GradientBoostingSurvivalAnalysis(random_state=12)

'''best_model = GradientBoostingSurvivalAnalysis(
    learning_rate = 0.2, 
    max_depth = 1, 
    min_samples_leaf = 1, 
    min_samples_split = 2, 
    n_estimators = 200,
    random_state=12
)'''

best_model = GradientBoostingSurvivalAnalysis(
    learning_rate = 0.2, 
    max_depth = 3, 
    min_samples_leaf = 5, 
    min_samples_split = 2, 
    n_estimators = 200,
    random_state=12
    )

In [9]:
grid_search = GridSearchCV(
    estimator=gbm,
    param_grid=param_grid,
    cv=cv,
    scoring=score_survival_model,
    n_jobs=-1,
    verbose=1
)

In [10]:
grid_search.fit(x_train, y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


GridSearchCV(cv=<generator object _BaseKFold.split at 0x0000018108D56E60>,
             estimator=GradientBoostingSurvivalAnalysis(random_state=12),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [1, 3, 5], 'min_samples_leaf': [1, 5],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring=<function score_survival_model at 0x000001810854CF40>,
             verbose=1)

In [11]:
print("Best parameters:", grid_search.best_params_)
print("Best C-Index:", grid_search.best_score_)
best_model = grid_search.best_estimator_

Best parameters: {'learning_rate': 0.2, 'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best C-Index: 0.837520096717358


In [45]:
results = train_evaluate(subsampling, x_train, y_train, sgkf, best_model)
results

ValueError: censoring survival function is zero at one or more time points

In [36]:
model  = GradientBoostingSurvivalAnalysis(
    learning_rate = 0.2, 
    max_depth = 3, 
    min_samples_leaf = 5, 
    min_samples_split = 2, 
    n_estimators = 200,
    random_state=12
    )

x_train, y_train, _ = subsample(subsampling, 
                        'Regional LN recurrence',
                        'Survival days', 
                        n_controls_per_case=5,
                        feature_columns=feature_columns)
model.fit(x_train, y_train)
predicted_risk = model.predict(x_test)
score = concordance_index_ipcw(y_train, y_test, predicted_risk)
score

(np.float64(0.6853897690477229),
 np.int64(1442),
 np.int64(512),
 np.int64(0),
 np.int64(2))